# Corrective RAG System
Answers questions from documents, checks the retrieved chunks, rewrites the question and searches again when the chunks are weak. On Kaggle turn on Internet in the notebook settings and add the novatech_data folder (and eval_questions.csv) with Add Data. On Colab upload them to the Files panel.

## 1. Install libraries

In [1]:
!pip install -q chromadb sentence-transformers pypdf python-docx groq gradio rank-bm25

print('installed')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
installed


## 2. Imports

In [2]:
import os
import re
import glob
import time
import getpass
import pandas as pd
import chromadb
import gradio as gr
from pypdf import PdfReader
from docx import Document
from groq import Groq
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

print('imports done')

imports done


## 3. Settings

In [3]:
chunk_size = 700
table_chunk_size = 1200
candidates = 15
rerank_keep = 6
embed_model_name = 'BAAI/bge-base-en-v1.5'
rerank_model_name = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
llm_model = 'openai/gpt-oss-120b'
# bge models need this prefix on the question only
query_prefix = 'Represent this sentence for searching relevant passages: '
no_answer = 'I could not find the answer in the documents.'

print(embed_model_name, rerank_model_name, llm_model)

BAAI/bge-base-en-v1.5 cross-encoder/ms-marco-MiniLM-L-6-v2 openai/gpt-oss-120b


## 4. Find the files

In [30]:
data_folder = None
for folder in ['novatech_data', '/content/novatech_data', '/kaggle/input']:
    if os.path.exists(folder):
        data_folder = folder
        break

all_files = glob.glob(data_folder + '/**/*.*', recursive=True)
file_paths = sorted([p for p in all_files if p.lower().endswith(('.pdf', '.docx', '.txt', '.csv')) and 'eval_questions' not in p])
eval_paths = [p for p in all_files if 'eval_questions' in p] + glob.glob('eval_questions.csv') + glob.glob('/content/eval_questions.csv')

print(data_folder, len(file_paths))
for path in file_paths:
    print(path)
print(eval_paths)

/kaggle/input 7
/kaggle/input/datasets/shrouqmansy/corrective-rag/Employee_handbook.docx
/kaggle/input/datasets/shrouqmansy/corrective-rag/HR_policy.pdf
/kaggle/input/datasets/shrouqmansy/corrective-rag/IT_security_policy.txt
/kaggle/input/datasets/shrouqmansy/corrective-rag/Offices_directory.csv
/kaggle/input/datasets/shrouqmansy/corrective-rag/Pricing.csv
/kaggle/input/datasets/shrouqmansy/corrective-rag/Product_guide.pdf
/kaggle/input/datasets/shrouqmansy/corrective-rag/Support_FAQ.txt
['/kaggle/input/datasets/shrouqmansy/questions/eval_questions.csv']


## 5. Text extraction and cleaning

In [32]:
def read_pdf(path):
    reader = PdfReader(path)
    return '\n'.join(page.extract_text() or '' for page in reader.pages)


def read_docx(path):
    doc = Document(path)
    lines = []
    for p in doc.paragraphs:
        if not p.text.strip():
            continue
        if p.style.name.startswith('Heading'):
            lines.append('# ' + p.text.strip())
        else:
            lines.append(p.text.strip())
    return '\n'.join(lines)


def read_txt(path):
    with open(path, encoding='utf-8') as file:
        return file.read()


def read_csv(path):
    df = pd.read_csv(path, dtype=str)
    rows = []
    for _, row in df.iterrows():
        rows.append('; '.join([col + ': ' + row[col] for col in df.columns if pd.notna(row[col])]) + '.')
    return '\n'.join(rows)


def clean_text(text):
    text = text.replace('\r', '')
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


readers = {'pdf': read_pdf, 'docx': read_docx, 'txt': read_txt, 'csv': read_csv}

print(list(readers))

['pdf', 'docx', 'txt', 'csv']


## 6. Load all documents

In [33]:
documents = []
for path in file_paths:
    extension = path.split('.')[-1].lower()
    file_name = os.path.basename(path)
    text = clean_text(readers[extension](path))
    documents.append({'source': file_name, 'type': extension, 'text': text})

print(pd.Series([d['type'] for d in documents]).value_counts())
print(sum(len(d['text']) for d in documents))

pdf     2
txt     2
csv     2
docx    1
Name: count, dtype: int64
12185


## 7. Chunking by section

In [34]:
def is_heading(line):
    line = line.strip()
    if line.startswith('# ') or line.startswith('Q:'):
        return True
    return bool(re.match(r'^\d+\.\s', line)) and len(line) < 60


def split_sections(text):
    sections = []
    title = 'Introduction'
    body = []
    for line in text.split('\n'):
        if not line.strip():
            continue
        if is_heading(line):
            if body:
                sections.append((title, ' '.join(body)))
            title = line.strip().lstrip('# ').strip()
            body = []
        else:
            body.append(line.strip())
    if body:
        sections.append((title, ' '.join(body)))
    return sections


def split_long(body, size):
    sentences = re.split(r'(?<=[.!?])\s+', body)
    pieces = []
    current = ''
    for s in sentences:
        if current and len(current) + len(s) > size:
            pieces.append(current)
            current = s
        else:
            current = current + ' ' + s if current else s
    if current:
        pieces.append(current)
    return pieces


def make_chunks(doc):
    result = []
    if doc['type'] == 'csv':
        current = ''
        for row in doc['text'].split('\n'):
            if current and len(current) + len(row) > table_chunk_size:
                result.append(('Table', current))
                current = row
            else:
                current = current + '\n' + row if current else row
        if current:
            result.append(('Table', current))
        return result
    for title, body in split_sections(doc['text']):
        if len(body) < 40:
            continue
        for piece in split_long(body, chunk_size):
            result.append((title, piece))
    return result


chunks = []
for doc in documents:
    for title, piece in make_chunks(doc):
        text = 'Source: ' + doc['source'] + '\nSection: ' + title + '\n' + piece
        chunks.append({'id': str(len(chunks)), 'text': text, 'source': doc['source'], 'section': title})

print(len(chunks))
print(chunks[5]['text'])

47
Source: Employee_handbook.docx
Section: Training budget
Each employee has a training budget of 15,000 EGP per year, subject to manager approval. An employee who leaves within 12 months after a company-paid certification must repay its cost.


## 8. Embeddings, vector database and keyword index

In [35]:
embedder = SentenceTransformer(embed_model_name)
texts = [c['text'] for c in chunks]
embeddings = embedder.encode(texts, batch_size=32, normalize_embeddings=True)

client = chromadb.Client()
try:
    client.delete_collection('docs')
except Exception:
    pass
collection = client.create_collection('docs')
collection.add(ids=[c['id'] for c in chunks], documents=texts, embeddings=embeddings.tolist(), metadatas=[{'source': c['source'], 'section': c['section']} for c in chunks])


def tokenize(text):
    return re.findall(r'[a-z0-9]+', text.lower())


bm25 = BM25Okapi([tokenize(t) for t in texts])

print(collection.count(), embeddings.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

47 (47, 768)


## 9. Retriever (vector search + keyword search + reranking)

In [36]:
reranker = CrossEncoder(rerank_model_name)


def retrieve(query):
    query_vector = embedder.encode(query_prefix + query, normalize_embeddings=True).tolist()
    dense = collection.query(query_embeddings=[query_vector], n_results=min(candidates, len(chunks)))
    dense_ids = [int(i) for i in dense['ids'][0]]
    scores = bm25.get_scores(tokenize(query))
    keyword_ids = [i for i in sorted(range(len(chunks)), key=lambda i: scores[i], reverse=True)[:candidates] if scores[i] > 0]
    fused = {}
    for ranking in [dense_ids, keyword_ids]:
        for rank, i in enumerate(ranking):
            fused[i] = fused.get(i, 0) + 1 / (60 + rank)
    best = sorted(fused, key=fused.get, reverse=True)[:candidates]
    rerank_scores = reranker.predict([[query, chunks[i]['text']] for i in best])
    ranked = sorted(zip(rerank_scores, best), key=lambda x: x[0], reverse=True)
    return [{'score': float(s), 'index': i, 'text': chunks[i]['text'], 'source': chunks[i]['source'], 'section': chunks[i]['section']} for s, i in ranked[:rerank_keep]]


hits = retrieve('How many vacation days do I get?')
for h in hits:
    print(round(h['score'], 2), h['source'], '|', h['section'])

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

-1.41 Employee_handbook.docx | Requesting time off
-2.4 HR_policy.pdf | 3. Annual leave
-5.2 HR_policy.pdf | 4. Sick leave
-5.95 HR_policy.pdf | 2. Probation period
-6.15 HR_policy.pdf | 5. Maternity and paternity leave
-7.24 Employee_handbook.docx | Business travel and expenses


## 10. Connect the LLM

In [38]:
if 'llm' not in globals():
    llm = Groq(api_key=getpass.getpass('Groq API key: '))


def ask_llm(system, user):
    for attempt in range(5):
        try:
            response = llm.chat.completions.create(model=llm_model, messages=[{'role': 'system', 'content': system}, {'role': 'user', 'content': user}], temperature=0, max_tokens=2000)
            return (response.choices[0].message.content or '').strip()
        except Exception as error:
            print('llm error, trying again:', str(error)[:100])
            time.sleep(20)
    return ''


print(ask_llm('Reply with one word.', 'Say hello'))

Hello


## 11. Grade the retrieved chunks

In [40]:
def grade_chunks(question, hits):
    passages = '\n\n'.join(['[' + str(i + 1) + '] ' + h['text'] for i, h in enumerate(hits)])
    system = 'You check retrieved passages for a question answering system. A passage is relevant if it contains facts that help answer the question, even if it uses different words than the question (for example vacation and annual leave mean the same thing). Reply with only the numbers of the relevant passages separated by commas. If no passage helps, reply with only the word NONE.'
    reply = ask_llm(system, 'Question: ' + question + '\n\nPassages:\n' + passages)
    numbers = sorted(set([int(n) for n in re.findall(r'\d+', reply)]))
    return [hits[n - 1] for n in numbers if 1 <= n <= len(hits)]

## 12. Rewrite the question

In [41]:
def rewrite_query(question):
    system = 'You help a search system. The first search for the question found nothing useful. Write 3 different search queries that could find the answer in company documents. Use other words the documents might use, such as synonyms, formal terms and related terms. Write one query per line with no numbering and no extra text.'
    reply = ask_llm(system, 'Question: ' + question)
    lines = [re.sub(r'^\s*(\d+[.)]|[-*])\s*', '', l).strip() for l in reply.split('\n') if l.strip()]
    return lines[:3]

## 13. Corrective RAG

In [42]:
def generate_answer(question, good):
    context = '\n\n'.join(['[' + str(i + 1) + '] ' + h['text'] for i, h in enumerate(good)])
    system = 'You answer questions about a company using only the context given. Match by meaning, not only by exact words. Give a short and direct answer and keep the exact numbers, dates and names from the context. If the context does not contain the answer, reply exactly: ' + no_answer + ' Never use outside knowledge and never mention passage numbers.'
    return ask_llm(system, 'Context:\n' + context + '\n\nQuestion: ' + question)


def search_again(question, hits):
    new_queries = rewrite_query(question)
    pool = {}
    for q in new_queries:
        for h in retrieve(q):
            if h['index'] not in pool or h['score'] > pool[h['index']]['score']:
                pool[h['index']] = h
    for h in hits:
        pool.setdefault(h['index'], h)
    merged = sorted(pool.values(), key=lambda h: h['score'], reverse=True)[:rerank_keep + 2]
    return grade_chunks(question, merged), new_queries


def corrective_answer(question):
    hits = retrieve(question)
    good = grade_chunks(question, hits)
    new_queries = []
    answer = ''
    if good:
        answer = generate_answer(question, good)
    if not good or 'could not find' in answer.lower():
        good, new_queries = search_again(question, hits)
        answer = generate_answer(question, good) if good else ''
    if not good or not answer or 'could not find' in answer.lower():
        return {'answer': no_answer, 'sources': [], 'rewritten': new_queries, 'good': len(good)}
    sources = sorted(set([h['source'] + ' / ' + h['section'] for h in good]))
    return {'answer': answer, 'sources': sources, 'rewritten': new_queries, 'good': len(good)}


result = corrective_answer('How many vacation days do I get?')
print(result['answer'])
print(result['sources'])
print(result['rewritten'])

result = corrective_answer('Is NovaDesk HIPAA compliant?')
print(result['answer'])
print(result['sources'])
print(result['rewritten'])

You receive 21 working days of paid annual leave per year, or 30 working days if you have more than 10 years of service.
['HR_policy.pdf / 3. Annual leave']
[]
I could not find the answer in the documents.
[]
['NovaDesk HIPAA compliance status', 'NovaDesk meets Health Insurance Portability and Accountability Act requirements', 'NovaDesk PHI security and regulatory compliance']


## 14. Plain RAG for comparison (no grading, no rewriting)

In [44]:
def plain_answer(question):
    hits = retrieve(question)[:4]
    answer = generate_answer(question, hits)
    if 'could not find' in answer.lower():
        return {'answer': no_answer, 'sources': [], 'rewritten': [], 'good': len(hits)}
    sources = sorted(set([h['source'] + ' / ' + h['section'] for h in hits]))
    return {'answer': answer, 'sources': sources, 'rewritten': [], 'good': len(hits)}

## 15. Evaluate the system

In [45]:
def normalize(text):
    text = text.lower().replace('**', '').replace(',', '')
    return re.sub(r'\s+', '', text)


eval_df = pd.read_csv(eval_paths[0], dtype=str, keep_default_na=False)


def run_eval(answer_function):
    rows = []
    for _, r in eval_df.iterrows():
        result = answer_function(r['question'])
        keywords = r['expected_keywords'].split('|')
        answer_ok = any([normalize(k) in normalize(result['answer']) for k in keywords])
        found = [s.split(' / ')[0] for s in result['sources']]
        expected = [s for s in r['expected_source'].split('|') if s]
        retrieval_ok = any([s in found for s in expected]) if expected else None
        rows.append({'id': r['id'], 'type': r['type'], 'question': r['question'], 'answer_ok': answer_ok, 'retrieval_ok': retrieval_ok, 'rewritten': len(result['rewritten']) > 0, 'answer': result['answer']})
        time.sleep(2)
    return pd.DataFrame(rows)


print(len(eval_df), eval_df['type'].value_counts().to_dict())

26 {'direct': 10, 'needs_rewrite': 10, 'unanswerable': 6}


In [46]:
corrective_results = run_eval(corrective_answer)

print(corrective_results[['id', 'type', 'retrieval_ok', 'answer_ok', 'rewritten']])
print(corrective_results.groupby('type')['answer_ok'].mean().round(3))
print('Retrieval hit rate:', round(corrective_results['retrieval_ok'].dropna().astype(bool).mean() * 100, 1), '%')
print('Answer accuracy:', round(corrective_results['answer_ok'].mean() * 100, 1), '%')
print('Questions that needed a rewrite:', corrective_results['rewritten'].sum())

    id           type retrieval_ok  answer_ok  rewritten
0    1         direct         True       True      False
1    2         direct         True       True      False
2    3         direct         True       True      False
3    4         direct         True       True      False
4    5         direct         True       True      False
5    6         direct         True       True      False
6    7         direct         True       True      False
7    8         direct         True       True      False
8    9         direct         True       True      False
9   10         direct         True       True      False
10  11  needs_rewrite         True       True      False
11  12  needs_rewrite         True       True      False
12  13  needs_rewrite         True       True      False
13  14  needs_rewrite         True       True      False
14  15  needs_rewrite         True       True      False
15  16  needs_rewrite         True       True      False
16  17  needs_rewrite         T

In [47]:
failed = corrective_results[~corrective_results['answer_ok']]
for _, r in failed.iterrows():
    print(r['question'])
    print(r['answer'])
    print('---')

print(len(failed))

0


## 16. Compare with plain RAG

In [48]:
plain_results = run_eval(plain_answer)

summary = pd.DataFrame({'plain RAG': plain_results.groupby('type')['answer_ok'].mean(), 'corrective RAG': corrective_results.groupby('type')['answer_ok'].mean()})
summary.loc['all'] = [plain_results['answer_ok'].mean(), corrective_results['answer_ok'].mean()]

print((summary * 100).round(1))

               plain RAG  corrective RAG
type                                    
direct             100.0           100.0
needs_rewrite      100.0           100.0
unanswerable       100.0           100.0
all                100.0           100.0


## 17. Gradio app

In [49]:
def chat(question):
    if not question.strip():
        return 'Please type a question.', '', ''
    result = corrective_answer(question)
    steps = 'Relevant passages found: ' + str(result['good'])
    if result['rewritten']:
        steps += '\nFirst search was weak, rewritten as:\n' + '\n'.join(result['rewritten'])
    return result['answer'], '\n'.join(result['sources']), steps


demo = gr.Interface(fn=chat, inputs=[gr.Textbox(label='Your question', lines=2)], outputs=[gr.Markdown(label='Answer'), gr.Textbox(label='Sources'), gr.Textbox(label='Process', lines=4)], title='Corrective RAG Assistant', description='Ask a question about the company documents.')
demo.launch(share=True)

print('app started')

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://bff5b4190b602fbcc5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


app started
